In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os 
from tqdm import tqdm

In [34]:
IMAGE_SIZE = 160
BATCH_SIZE = 192

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [35]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,          # Increase batch size
    shuffle=True,
    num_workers=4,          # Parallel loading
    pin_memory=True         # Faster GPU transfer
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True
)

In [36]:
dataset = datasets.ImageFolder(root="dataset", transform=transform)

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(dataset.classes)  # ['negative', 'positive']

['Negative', 'Positive']


In [37]:
model = models.resnet18(pretrained=True)

# Freeze backbone
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
model.fc = nn.Linear(model.fc.in_features, 2)

model = model.to(device)

In [38]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)

In [39]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    for images, labels in tqdm(train_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_acc = 100 * correct / total

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    print(f"Train Loss: {running_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Acc: {val_acc:.2f}%")
    print("-" * 40)


Epoch 1/10


100%|██████████| 42/42 [01:09<00:00,  1.65s/it]


Train Loss: 26.1272 | Train Acc: 95.27%
Val Acc: 97.94%
----------------------------------------

Epoch 2/10


100%|██████████| 42/42 [00:18<00:00,  2.25it/s]


Train Loss: 11.6732 | Train Acc: 98.02%
Val Acc: 98.39%
----------------------------------------

Epoch 3/10


...

Train Loss: 6.4003 | Train Acc: 98.72%
Val Acc: 98.64%
----------------------------------------

Epoch 9/10


100%|██████████| 42/42 [00:48<00:00,  1.16s/it]


Train Loss: 6.6326 | Train Acc: 98.53%
Val Acc: 98.69%
----------------------------------------

Epoch 10/10


100%|██████████| 42/42 [00:49<00:00,  1.19s/it]

Train Loss: 6.6487 | Train Acc: 98.66%
Val Acc: 98.85%
----------------------------------------


In [25]:



model = torch.compile(model)

In [40]:
torch.save(model.state_dict(), "crack_model.pth")

In [41]:
from PIL import Image

def predict_image(image_path):
    model.eval()
    image = Image.open(image_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, predicted = torch.max(outputs, 1)

    class_names = dataset.classes
    print("Prediction:", class_names[predicted.item()])

predict_image("dataset/positive/example.jpg")

FileNotFoundError: [Errno 2] No such file or directory: 'dataset/positive/example.jpg'

In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")